# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)

(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [10]:
# Q1 — Relative population growth by continent
continent_pop = df_long.groupby(['Year', 'Continent'])['Population'].sum().reset_index()

base = continent_pop[continent_pop['Year'] == 1970][['Continent', 'Population']]
base = base.rename(columns={'Population': 'Base'})

continent_pop = continent_pop.merge(base, on='Continent')
continent_pop['Growth (%)'] = (continent_pop['Population'] / continent_pop['Base'] - 1) * 100

fig = px.line(continent_pop, x='Year', y='Growth (%)',
              color='Continent',
              title='Population Growth by Continent Since 1970',
              template='plotly_white',
              color_discrete_sequence=px.colors.qualitative.Safe)

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

I selected a line chart to compare population growth over different years. By setting the 1970 population as the starting point, we can see which continent experienced the highest relative growth.

---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [12]:
# Q2 — Top 15 countries by absolute population added, 1970–2022
df['Increase'] = df['2022 Population'] - df['1970 Population']

top15 = df.nlargest(15, 'Increase').sort_values('Increase')

top15['Group'] = top15['Continent'].apply(
    lambda x: 'Africa' if x == 'Africa' else 'Other'
)

fig = px.bar(top15, x='Increase', y='Country',
             orientation='h', color='Group',
             color_discrete_map={'Africa': '#0072B2', 'Other': '#999999'},
             title='Countries with the Largest Population Increase',
             template='plotly_white')

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()


I chose a horizontal bar chart to compare population increases across countries. Sorting the values makes it easier to identify the countries that added the largest number of people.


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [13]:
# Q3 — Land area versus 2022 population
fig = px.scatter(df, x='Area', y='2022 Population',
                 color='Continent', hover_name='Country',
                 log_x=True, log_y=True,
                 title='Relationship Between Land Area and Population',
                 template='plotly_white',
                 color_discrete_sequence=px.colors.qualitative.Safe)

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

I used a scatter plot to examine whether countries with larger land areas also have larger populations. The logarithmic scales help compare countries with very different sizes and population levels.


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [14]:
# Q4 — Distribution of population density by continent
fig = px.violin(df, x='Continent', y='Density',
                color='Continent', box=True,
                title='Distribution of Population Density by Continent',
                template='plotly_white',
                color_discrete_sequence=px.colors.qualitative.Safe)

fig.update_layout(title_x=0.5, showlegend=False)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)

fig.show()

I selected a violin plot to compare population density across continents. It shows both the spread and shape of the distributions, helping us understand how population density varies between countries

---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [15]:
# Q5 — Geographic pattern of population change, 2000–2022
df['Change (%)'] = (
    (df['2022 Population'] - df['2000 Population']) /
    df['2000 Population'] * 100
)

fig = px.choropleth(df, locations='CCA3',
                    locationmode='ISO-3',
                    color='Change (%)',
                    hover_name='Country',
                    color_continuous_scale='RdBu',
                    color_continuous_midpoint=0,
                    title='Population Growth and Decline Across Countries',
                    template='plotly_white')

fig.update_layout(title_x=0.5)

fig.show()

I chose a choropleth map to display population changes across countries. The different shades show the percentage change, making it easier to identify areas with population growth or decline.

---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [16]:
# Q6 — 2022 population by continent and country
fig = px.treemap(df, path=['Continent', 'Country'],
                 values='2022 Population',
                 color='Continent',
                 color_discrete_sequence=px.colors.qualitative.Safe,
                 title='Distribution of World Population in 2022',
                 template='plotly_white')

fig.update_layout(title_x=0.5)

fig.show()

I used a treemap to represent the population distribution across continents and countries. The size of each rectangle represents the population, allowing us to compare countries within their continents.

---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

*YOUR ANSWER HERE*

In [17]:
# Q7 — World population shares over time
pop = df_long.groupby(['Year', 'Continent'])['Population'].sum().reset_index()

pop['Continent'] = pop['Continent'].replace({
    'North America': 'Rest of the world',
    'South America': 'Rest of the world',
    'Oceania': 'Rest of the world'
})

pop = pop.groupby(['Year', 'Continent'])['Population'].sum().reset_index()

pop['Share'] = pop.groupby('Year')['Population'].transform(
    lambda x: x / x.sum() * 100
)

fig = px.line(pop, x='Year', y='Share', color='Continent',
              markers=True,
              title='How World Population Shares Changed Over Time',
              template='plotly_white',
              color_discrete_sequence=px.colors.qualitative.Safe)

fig.update_layout(title_x=0.5)
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False, title='Population Share (%)')

fig.show()


I used a line chart to show how the population share of Africa, Asia, Europe and the rest of the world changes over time. This makes it easier to compare the trends and see which regions are gaining or losing their share.